Using POS-Tagging, Dependency-Parsing on Samsung reviews to find the qualitative aspects of top 5 / 10 features

In [308]:
import numpy as np
import pandas as pd
import re
from tqdm import tqdm
import spacy

In [309]:
path = r'C:\Users\arnig\Documents\Coding_2024\python_work\UpGrad\DS_C70\Specialization_NLP\Syntactic-Processing\Syntactic-Processing_upgrad\POS Tagging Case Study\Dataset\Samsung.txt'
with open(path, mode='r', encoding='utf-8') as f_open:
    reviews_data = f_open.read()

print(type(reviews_data))
print(len(reviews_data))

<class 'str'>
7488235


In [310]:
# Splitting into sentences
reviews = reviews_data.split('\n')
reviews[:10]

["I feel so LUCKY to have found this used (phone to us & not used hard at all), phone on line from someone who upgraded and sold this one. My Son liked his old one that finally fell apart after 2.5+ years and didn't want an upgrade!! Thank you Seller, we really appreciate it & your honesty re: said used phone.I recommend this seller very highly & would but from them again!!",
 'nice phone, nice up grade from my pantach revue. Very clean set up and easy set up. never had an android phone but they are fantastic to say the least. perfect size for surfing and social media. great phone samsung',
 'Very pleased',
 'It works good but it goes slow sometimes but its a very good phone I love it',
 'Great phone to replace my lost phone. The only thing is the volume up button does not work, but I can still go into settings to adjust. Other than that, it does the job until I am eligible to upgrade my phone again.Thaanks!',
 'I originally was using the Samsung S2 Galaxy for Sprint and wanted to retu

In [311]:
# Top features
nlp_pos = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
nouns = []
for doc in tqdm(reviews):
    tokens = nlp_pos(doc)
    nouns.extend(token.lemma_ for token in tokens if token.pos_ == 'NOUN')

nouns = pd.Series(nouns).value_counts(normalize= True)

100%|██████████| 46355/46355 [02:16<00:00, 340.02it/s]


In [312]:
# Top 10 features from reviews
top = 10
features = nouns.head(top).index.values
features

array(['phone', 'battery', 'product', 'time', 'screen', 'card', 'price',
       'problem', 'camera', 'app'], dtype=object)

In [313]:
# Filtering reviews based on presence of top feature words
feature_reviews = {feature: [doc for doc in reviews if feature in doc] for feature in features}

Filtering qualities based on pos and dependency of qualifiers with feature words.

In [314]:
nlp_dep = spacy.load('en_core_web_sm', disable=['ner'])
feature_qualities = {}

for feature, docs in feature_reviews.items():
    qualities = []
    for doc in tqdm(docs):
        for tok in nlp_dep(doc):

            if tok.pos_ == 'ADV' and feature in [str(a) for a in list(tok.ancestors)]:
                adv = tok.lemma_
            else:
                adv = ''
            if tok.pos_ == 'ADJ' and feature in [str(a) for a in list(tok.ancestors)]:
                adj = tok.lemma_
            else:
                adj = ''
            quality = adv + adj

            if len(quality):
                qualities.append(quality)

    feature_qualities[feature] = pd.Series(qualities)

100%|██████████| 5061/5061 [00:52<00:00, 95.66it/s] 


Concat features and qualifiers into a dataframe

In [315]:
FeatureQuality = pd.concat([pd.Series(feature_qualities[key].value_counts(normalize= True).head(100), name=key) for key in feature_qualities.keys()], axis=1)

In [316]:
FeatureQuality[FeatureQuality.isna()] = 0
FeatureQuality

,phone,battery,product,time,screen,card,price,problem,camera,app
great,0.164017,0.021298,0.209770,0.016434,0.053181,0.002534,0.250273,0.000000,0.119748,0.030702
good,0.110307,0.028398,0.228448,0.027938,0.047714,0.005912,0.200109,0.009379,0.111345,0.021930
new,0.048410,0.087221,0.011135,0.018077,0.003976,0.070101,0.006543,0.018757,0.002801,0.013158
very,0.045855,0.017241,0.079023,0.031224,0.038270,0.000000,0.050709,0.010551,0.043417,0.000000
nice,0.045239,0.012170,0.020833,0.003287,0.043241,0.000000,0.016358,0.001172,0.035714,0.000000
...,...,...,...,...,...,...,...,...,...,...
comparable,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.004386
thick,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.004386
pary,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.004386
saver,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.004386


Feature importance by vector length

In [317]:
feature_imp = {}
for col in FeatureQuality.columns:
    feature_imp[col] = np.linalg.norm(FeatureQuality[col].values)

for key in sorted(feature_imp, key= lambda x: feature_imp[x], reverse= True):
    print(f'{key} = {feature_imp[key]}')

product = 0.3910338146778301
problem = 0.35208263658641564
price = 0.3349542598790246
card = 0.23371017529678392
phone = 0.22551300339676997
time = 0.22425587676410486
camera = 0.21136862162368047
battery = 0.1797737076553386
screen = 0.17884634832089885
app = 0.17389673334735972


Frobenius norm of Feature Quality matrix

In [318]:
F = FeatureQuality.values
print('Frobenius norm: ', np.linalg.norm(F))

Frobenius norm:  0.8270747322749079


Cosine similarity between features in the context of feature qualifiers

In [319]:
start_pos = 0
similarities = np.zeros((F.shape[1], F.shape[1]))
for i in range(F.shape[1]):
    for j in range(F.shape[1]):
        if i != j:
            s = np.dot(F[:, i], F[:, j]) / (np.linalg.norm(F[:, i]) * np.linalg.norm(F[:, j]))
            similarities[i][j] = s
        else:
            similarities[i][j] = 0
    start_pos += 1

Maximum and minimum cosine similarities

In [320]:
max_val = similarities[0][0]
min_val = similarities[0][0]

for i in range(similarities.shape[0]):
    for j in range(similarities.shape[1]):

        if max_val < similarities[i][j]:
            max_idx = [i, j]
            max_val = similarities[i][j]

        if similarities[i][j] != 0 and min_val > similarities[i][j]:
            min_idx = [i, j]
            min_val = similarities[i][j]

print(f'Max cosine similarity: {FeatureQuality.columns[max_idx[0]]}, {FeatureQuality.columns[max_idx[1]]}')
print(f'Min cosine similarity: {FeatureQuality.columns[min_idx[0]]}, {FeatureQuality.columns[min_idx[1]]}')

Max cosine similarity: phone, price
Min cosine similarity: time, product


Qualifier words for each feature type

In [321]:
for col in FeatureQuality.columns:
    print(col)
    print(FeatureQuality.loc[FeatureQuality[col] != 0, col].sort_values(ascending=False).index.values)
    print('-'*80)

phone
['great' 'good' 'new' 'very' 'nice' 'smart' 'excellent' 'unlocked' 'old'
 'ever' 'awesome' 'first' 'amazing' 'really' 'flip' 'well' 'just' 'best'
 'basic' 'fast' 'other' 'little' 'previous' 'perfect' 'android' 'same'
 'so' 'beautiful' 'small' 'simple' 'last' 'overall' 'cheap'
 'international' 'dual' 'far' 'big' 'easy' 'wonderful' 'fantastic' 'solid'
 'only' 'more' 'second' 'high' 'mobile' 'right' '-' 'much' 'pretty' 'used'
 'large' 'decent' 'expensive' 'cool' 'reliable' 'rugged' 'super' 'still'
 'most' 'out' 'even' 'different' 'light' 'durable' 'low' 'there' 'prepaid'
 'otherwise' 'non' 'own' 'real' 'particular' 'too' 'outstanding'
 'favorite' 'starter' 'unlock' 'tough' 'mid' 'lovely' 'regular' 'all'
 'original' 'absolutely' 'straight' 'bad' 'long' 'powerful' 'happy' 'mini'
 'free' 'late' 'sturdy' 'as' 'actual' 'now' 'current' 'many' 'fully']
--------------------------------------------------------------------------------
battery
['removable' 'new' 'long' 'replaceable' 'good' 'bi

Dependencies are not analysed if spacy parser is disabled